In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [3]:
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(vocab_size)
print(itos)

27
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
block_size = 3
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1]) # training set: 80%
Xdev, Ydev = build_dataset(words[n1:n2]) # validation set : 10%
Xte, Yte = build_dataset(words[n2:]) # test set : 10%


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    macdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approx: {str(app):5s} | maxdiff: {macdiff}')
    

just copy pasted and edited a bit from previous book

In [7]:
# mlp
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((vocab_size, n_embd), generator=g) 
# the setting of scaling factor is very important (mostly because we are only using gradient descent, and not using any fancy optimizer like Adam or RMSProp)
# the best way proposed in the paper mentioned and torch docs is 
# to multiply the weights by (5/3)/ (fan_in)**0.5, where fan_in in our case is block_size * n_embd, and the factor 5/3 is a magic number that works well in practice.
W1 = torch.randn((block_size * n_embd, n_hidden), generator = g) * (5/3) /( (block_size * n_embd)**0.5) #* 0.2
# the above thing is done to get the standard deviation of the weights to be 1, which is a good practice for training neural networks, and also to make sure that the weights are not too large, which can cause the gradients to explode and the training to diverge.
# the above methodology is not very much used today, due to the use of optimizers and normalization techniques like batch normalization
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

# Batch normalization parameters
bngain = torch.ones((1, n_hidden)) *0.1 + 0.1
bnbias = torch.zeros((1, n_hidden))* 0.1
# bnmean_running = torch.zeros((1, n_hidden))
# bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1,b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

4137


In [8]:
batch_size = 32
n = batch_size 
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [9]:
emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors into

# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation

# Batch normalization
bnmeani = 1 /n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*bndiff2.sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain* bnraw + bnbias

# Non-linearity
h = torch.tanh(hpreact) # hidden layer post-activation
# cross-entropy loss
logits = h @ W2 + b2
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # if 1/counts_sum, then we don't get backprop to be exact bit
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[torch.arange(n), Yb].mean()

# Pytorch backward pass
for p in parameters:
    p.grad = None

for t in [
    logprobs, probs,counts, counts_sum, counts_sum_inv,
    norm_logits, logit_maxes, logits,h, hpreact, bnraw, bnvar_inv,
    bnvar, bndiff2, bndiff,hprebn, bnmeani, embcat, emb
]:
    t.retain_grad()

loss.backward()
loss

tensor(3.2848, grad_fn=<NegBackward0>)

In [10]:
# logits.shape, logit_maxes.shape
# bndiff.shape, bndiff2.shape, bnmeani.shape
# hprebn.shape, bnmeani.shape
# hprebn.shape, embcat.shape, W1.shape, b1.shape
C.shape, Xb.shape, emb.shape
# emb.sum(1).shape

(torch.Size([27, 10]), torch.Size([32, 3]), torch.Size([32, 3, 10]))

In [11]:
# Excercise 1 backprop through the whole thing manually
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n

dprobs = (1.0 / probs) * dlogprobs

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = (counts_sum_inv * dprobs)

dcounts_sum = (-1 * counts_sum**-2)*dcounts_sum_inv
# since the counts was used by two branches, therefore we add both the gradients.
# also since the counts_sum was just addition of the rows, and we know derivative is just passed on directly from the prev branch when added, therefoe below.
dcounts += torch.ones_like(counts)*dcounts_sum
dnorm_logits =  (norm_logits).exp() * dcounts
dlogits = dnorm_logits.clone()
dlogits_max = -1* dnorm_logits.sum(1, keepdim=True) 
# dlogits += torch.zeros_like(logits) * dlogits_max # this worked, but was not eact. the bottom implementation works better
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]).float() * dlogits_max

# Theres a good derivation of it, but a hacky way of doing it is to just compare the shapes and figure out what works on mat mul.
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h**2) * dh #idk why i'm not getting an exact true

dbngain = (bnraw * dhpreact).sum(0, keepdim = True)
dbnraw = (bngain * dhpreact)
dbnbias = dhpreact.sum(0, keepdim=True)

dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim= True)

dbnvar = -0.5 *( (bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = 1.0/(n-1) * torch.ones_like(bndiff2) * dbnvar
dbndiff += 2.0 * bndiff * dbndiff2

dhprebn = dbndiff.clone()
dbnmeani = (- dbndiff ).sum(0, keepdim=True)
dhprebn +=( 1.0/n) * torch.ones_like(hprebn) * dbnmeani

dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

# the operation was just about changing the view of the tensor, therefore changing it back to original shape is just differntition lol
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[i, j]
        dC[ix] += demb[i,j]

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogits_max, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnraw', dbnraw, bnraw)
cmp('bnbias', dbnbias, bnbias)
cmp('bndiff', dbndiff, bndiff)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approx: True  | maxdiff: 0.0
probs           | exact: True  | approx: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approx: True  | maxdiff: 0.0
counts_sum      | exact: True  | approx: True  | maxdiff: 0.0
counts          | exact: True  | approx: True  | maxdiff: 0.0
norm_logits     | exact: True  | approx: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approx: True  | maxdiff: 0.0
logits          | exact: True  | approx: True  | maxdiff: 0.0
h               | exact: True  | approx: True  | maxdiff: 0.0
W2              | exact: True  | approx: True  | maxdiff: 0.0
b2              | exact: True  | approx: True  | maxdiff: 0.0
hpreact         | exact: False | approx: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approx: True  | maxdiff: 1.862645149230957e-09
bnraw           | exact: False | approx: True  | maxdiff: 1.1641532182693481e-10
bnbias          | exact: False | approx: True  | maxdiff: 3.725290298461914e-

In [12]:
# Excercise 2: backprop through cross_entropy, but all in one go
# forward pass
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff: ', (loss_fast - loss).item())

3.284794330596924 diff:  0.0


In [13]:
# backward pass
# after derivin, we get if y = i, loss = P_i - 1, if not then P_i. wheree P_i is just softmax for ith element.
dlogits = F.softmax(logits,1 )
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)
dlogits.shape

logits          | exact: False | approx: True  | maxdiff: 5.820766091346741e-09


torch.Size([32, 27])

In [14]:
# excercise 3: backprop through batchnorm but all in one go
# forward pass
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased = True)+1e-5) + bnbias
print("max_diff: ", (hpreact_fast - hpreact).abs().max())

max_diff:  tensor(8.9407e-08, grad_fn=<MaxBackward1>)


In [15]:
hprebn.shape, dlogits.shape, 

(torch.Size([32, 64]), torch.Size([32, 27]))

In [16]:
# backward pass
# calculate hprbn given hpreact.
# we'll ignore gamma and beta from the equation
dhprebn = (bngain * bnvar_inv / n) * (n * dhpreact - dhpreact.sum(0) - (n/(n-1))*bnraw*(dhpreact * bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approx: True  | maxdiff: 2.3283064365386963e-10


In [24]:
# Excercise 4
# Putting it all together

# mlp
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((vocab_size, n_embd), generator=g) 
W1 = torch.randn((block_size * n_embd, n_hidden), generator = g) * (5/3) /( (block_size * n_embd)**0.5) #* 0.2
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

# Batch normalization parameters
bngain = torch.ones((1, n_hidden)) *0.1 + 0.1
bnbias = torch.zeros((1, n_hidden))* 0.1

parameters = [C, W1,b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

with torch.no_grad():
    for i in range(max_steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
        Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

        emb = C[Xb] # embed the characters into vectors
        embcat = emb.view(emb.shape[0], -1) # concatenate the vectors

    # Linear layer 
        hprebn = embcat @ W1 + b1 # hidden layer pre-activation

        # Batch norm layer
        bnmean = hprebn.mean(0, keepdim=True)
        bnvar = hprebn.var(0, keepdim=True, unbiased=True)
        bnvar_inv = (bnvar + 1e-5)**-0.5
        bnraw = (hprebn - bnmean) * bnvar_inv
        hpreact = bngain * bnraw + bnbias

        # non linearity
        h = torch.tanh(hpreact) # hidden layer post-activation
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Yb)

        # backward pass
        for p in parameters:
            p.grad = None

        # loss.backward()

        # Manual backpropagation 

        dlogits = F.softmax(logits, 1)
        dlogits[range(n), Yb] -= 1
        dlogits /= n
        # 2nd layer backprop
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)
        # tanh
        dhpreact = (1.0 - h**2) * dh
        # batchnorm backprop
        dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
        dbnbias = dhpreact.sum(0, keepdim=True)
        dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
        # 1st layer
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(0)
        # embedding
        demb = dembcat.view(emb.shape)
        dC = torch.zeros_like(C)
        for k in range(Xb.shape[0]):
            for j in range(Xb.shape[1]):
                ix = Xb[k,j]
                dC[ix] += demb[k,j]
        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
        # update
        lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
        for p, grad in zip(parameters, grads):
            # p.data += -lr * p.grad
            p.data += -lr * grad

        # track stats
        if i % 10000 == 0:
            print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
        lossi.append(loss.log10().item())

        # if i >=100:
        #     break

4137
      0/ 200000: 3.2848
  10000/ 200000: 2.4611
  20000/ 200000: 2.0706
  30000/ 200000: 2.3333
  40000/ 200000: 2.3128
  50000/ 200000: 2.2758
  60000/ 200000: 2.2560
  70000/ 200000: 2.4851
  80000/ 200000: 2.1520
  90000/ 200000: 2.4611
 100000/ 200000: 2.0781
 110000/ 200000: 1.9559
 120000/ 200000: 2.0598
 130000/ 200000: 2.4147
 140000/ 200000: 1.9316
 150000/ 200000: 2.1198
 160000/ 200000: 2.0866
 170000/ 200000: 2.2309
 180000/ 200000: 1.9228
 190000/ 200000: 2.0334


In [25]:
with torch.no_grad():
    emb = C[Xtr] # embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    hpreact = embcat @ W1 + b1 # hidden layer pre-activation
    bmean = hpreact.mean(0, keepdim=True)
    bvar = hpreact.var(0, keepdim=True, unbiased=True)

In [26]:

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.1904609203338623
val 2.207965135574341


In [27]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # ------------
      # forward pass:
      # Embedding
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # ------------
      # Sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carpaiza.
jahleigh.
mrex.
taty.
sacalene.
rah.
bedlee.
art.
kaeli.
nerisia.
chaiiv.
kaleig.
dell.
join.
quint.
shoul.
alia.
biyo.
jeron.
jaryxi.
